# Day 2 · Hands-on 4 — Logistic regression
**About 30 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)


When the outcome is yes/no (disease, response, death), a straight line can predict probabilities below 0 or above 1. **Logistic regression** models the log-odds instead, and its effects are **odds ratios**.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for the simulated dataset, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
# Every analysis below uses ROLES (outcome, group, ...), never hard-coded column names.
# To add a dataset: write a loader function and add one entry to DATASETS.
%pip install -q lifelines
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from lifelines.datasets import load_gbsg2, load_lung

def _gbsg2():
    d = load_gbsg2()
    for c in ["tgrade", "menostat", "horTh"]:
        d[c] = d[c].astype(str)
    d["nodes4"] = (d["pnodes"] >= 4).astype(int)
    return d

def _lung():
    d = load_lung()
    d.columns = [c.replace(".", "_") for c in d.columns]
    d = d.dropna(subset=["ph_ecog"])
    d["ecog"] = d["ph_ecog"].map({0: "0", 1: "1", 2: "2+", 3: "2+"})
    d["sex"] = d["sex"].map({1: "male", 2: "female"})
    d["lost_weight"] = np.where(d["wt_loss"].isna(), np.nan, (d["wt_loss"] > 0).astype(float))
    return d

def _simulated(seed=2026, n=300):
    rng = np.random.default_rng(seed)
    genotype = rng.choice(["AA", "AB", "BB"], size=n, p=[0.4, 0.4, 0.2])
    sex = rng.choice(["F", "M"], size=n)
    age = rng.uniform(30, 80, n) + np.where(genotype == "BB", 8, 0)          # BB carriers are older
    g_eff = pd.Series({"AA": 0, "AB": 3, "BB": 8})[genotype].to_numpy()
    slope = np.where(sex == "M", 0.7, 0.4)                                    # interaction: steeper in men
    biomarker = 50 + slope * (age - 50) + g_eff + rng.normal(0, 6, n)
    logit = -1 + 0.05 * (age - 50) + pd.Series({"AA": 0, "AB": 0.4, "BB": 1.0})[genotype].to_numpy()
    disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    mu = np.exp(0.5 + 0.02 * (age - 50) + pd.Series({"AA": 0, "AB": 0.3, "BB": 0.7})[genotype].to_numpy())
    alpha = 0.8
    lesions = rng.negative_binomial(1 / alpha, 1 / (1 + alpha * mu))
    return pd.DataFrame(dict(biomarker=biomarker.round(1), genotype=genotype, sex=sex, age=age.round(1),
                             disease=disease, lesions=lesions))

def _my_data():
    # Upload a CSV to Colab (folder icon on the left) and edit the roles in DATASETS["my_data"]
    return pd.read_csv("my_data.csv")

DATASETS = {
    "gbsg2": dict(load=_gbsg2,
        about="GBSG2 breast cancer trial: 686 patients",
        outcome="tsize", outcome_label="Tumour size (mm)",
        group="tgrade", group_label="Tumour grade",
        x="pnodes", x_label="Positive lymph nodes",
        group2="menostat", group2_label="Menopausal status",
        binary="nodes4", binary_label="4 or more positive lymph nodes",
        binary_x="tsize", binary_group="tgrade",
        count="pnodes", count_label="Positive lymph nodes", count_x="tsize", count_group="tgrade"),
    "lung": dict(load=_lung,
        about="NCCTG advanced lung cancer study: 227 patients",
        outcome="pat_karno", outcome_label="Patient-rated Karnofsky score",
        group="ecog", group_label="ECOG performance status",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="lost_weight", binary_label="Lost weight in last 6 months",
        binary_x="age", binary_group="ecog",
        count=None),
    "simulated": dict(load=_simulated,
        about="Simulated biomarker study: 300 people, true effects known",
        outcome="biomarker", outcome_label="Biomarker level",
        group="genotype", group_label="Genotype",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="disease", binary_label="Disease present",
        binary_x="age", binary_group="genotype",
        count="lesions", count_label="Number of lesions", count_x="age", count_group="genotype",
        truth={
            "group": "AB vs AA = +3, BB vs AA = +8 (adjusted for age). BB carriers are also ~8 years older.",
            "x": "age slope = 0.4 per year in women, 0.7 in men",
            "interaction": "difference in age slopes (M vs F) = +0.3 per year",
            "binary": "odds ratio per year of age = 1.05; AB vs AA = 1.49; BB vs AA = 2.72",
            "count": "rate ratio per year of age = 1.02; AB vs AA = 1.35; BB vs AA = 2.01; overdispersed (alpha = 0.8)",
        }),
    "my_data": dict(load=_my_data,
        about="Your own CSV: edit these roles to match your column names",
        outcome="my_outcome", outcome_label="My outcome",
        group="my_group", group_label="My group",
        x="my_numeric_predictor", x_label="My numeric predictor",
        group2="my_two_level_group", group2_label="My two-level group",
        binary="my_yes_no_outcome", binary_label="My yes/no outcome (coded 0/1)",
        binary_x="my_numeric_predictor", binary_group="my_group",
        count=None),
}

def load_dataset(name):
    R = dict(DATASETS[name])
    d = R.pop("load")()
    for role in ["group", "group2", "binary_group", "count_group"]:
        if R.get(role):
            d[R[role]] = d[R[role]].astype(str)
    R["levels"] = sorted(d[R["group"]].dropna().unique())
    R.setdefault("truth", {})
    print(f"✅ {R['about']}  ({len(d)} rows)")
    print(f"   outcome: {R['outcome']} ({R['outcome_label']})   group: {R['group']} {R['levels']}")
    print(f"   numeric predictor: {R['x']}   second group: {R['group2']}")
    return d, R

def cat(var, ref):
    """Categorical term with an explicit reference level."""
    return f"C({var}, Treatment(reference='{ref}'))"

def nice(term):
    """Turn statsmodels term names into readable labels."""
    m = re.fullmatch(r"C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\](.*)", term)
    if m:
        var, ref, lvl, rest = m.groups()
        label = f"{var}: {lvl} vs {ref}"
        return label + (f" × {rest.lstrip(':')}" if rest else "")
    m = re.fullmatch(r"(\w+):C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\]", term)
    if m:
        x, var, ref, lvl = m.groups()
        return f"{x} × {var}: {lvl} vs {ref}"
    return term

def tidy(fit, exponentiate=False):
    """Coefficient table: estimate, 95% CI, p-value."""
    ci = fit.conf_int()
    t = pd.DataFrame({"estimate": fit.params, "CI low": ci[0], "CI high": ci[1], "p-value": fit.pvalues})
    t = t.drop(index=[i for i in t.index if i == "alpha"], errors="ignore")
    if exponentiate:
        t[["estimate", "CI low", "CI high"]] = np.exp(t[["estimate", "CI low", "CI high"]])
    t.index = [nice(i) for i in t.index]
    return t.round({"estimate": 3, "CI low": 3, "CI high": 3}).assign(**{"p-value": t["p-value"].map(lambda p: f"{p:.2g}")})

def show_truth(R, key):
    if key in R["truth"]:
        print(f"🎯 Truth used to simulate: {R['truth'][key]}")

print("✅ Setup complete")

## Choose a dataset
Everything in this notebook is written in terms of **roles** (an outcome, a group, a numeric predictor), so the same cells work on any dataset. Start with the one your facilitator names, then try `simulated`: it was generated with known effects, so you can check whether the model recovers the truth.

In [ ]:
# 🔧 CHANGE THIS: which dataset to use (all cells below adapt automatically)
dataset = "gbsg2"  # @param ["gbsg2", "lung", "simulated", "my_data"]

df, R = load_dataset(dataset)
df.head()

## A. Why not a straight line?
Switch between the two models and look at the predictions at the extremes.

In [ ]:
# 🔧 CHANGE THIS: "linear" or "logistic"
model = "linear"

yb, x = R["binary"], R["binary_x"]
d = df.dropna(subset=[yb, x])
xs = pd.DataFrame({x: np.linspace(d[x].min(), d[x].max(), 100)})
fit = (smf.ols if model == "linear" else smf.logit)(f"{yb} ~ {x}", data=d).fit(**({} if model == "linear" else {"disp": 0}))
pred = fit.predict(xs)

plt.figure(figsize=(6, 3.5))
plt.scatter(d[x], d[yb] + np.random.uniform(-0.03, 0.03, len(d)), alpha=0.2, s=10)
plt.plot(xs[x], pred, color="black", linewidth=2.5)
plt.axhspan(1, 1.3, color="red", alpha=0.08); plt.axhspan(-0.3, 0, color="red", alpha=0.08)
plt.ylim(-0.3, 1.3); plt.xlabel(x); plt.ylabel(f"P({R['binary_label']})"); plt.title(f"{model} model")
plt.show()
print(f"Predicted probabilities range from {pred.min():.2f} to {pred.max():.2f}")

## B. Fit the model and read odds ratios
### 🧭 Concept
Exponentiated coefficients are **odds ratios (OR)**. For a numeric predictor: the multiplication of the odds per one-unit increase. For a group: the odds in that group relative to the reference.

In [ ]:
# 🔧 CHANGE THIS: reference group
reference = sorted(df[R["binary_group"]].dropna().unique())[0]

yb, x, g = R["binary"], R["binary_x"], R["binary_group"]
d = df.dropna(subset=[yb, x, g])
logit_fit = smf.logit(f"{yb} ~ {x} + {cat(g, reference)}", data=d).fit(disp=0)
print(f"Outcome: {R['binary_label']}  (overall {d[yb].mean():.0%} of rows)\n")
print("Odds ratios with 95% confidence intervals:")
display(tidy(logit_fit, exponentiate=True))
show_truth(R, "binary")

## C. From odds ratios to probabilities
### 🧭 Concept
The odds ratio for a group is the same at every value of the numeric predictor, but the **difference in probability** is not. Try a low, a middle and a high value.

In [ ]:
# 🔧 CHANGE THIS: value of the numeric predictor to predict at
x_value = float(df[R["binary_x"]].median())

g = R["binary_group"]
levels = sorted(d[g].unique())
newdata = pd.DataFrame({R["binary_x"]: x_value, g: levels})
probs = logit_fit.predict(newdata)
print(f"Predicted probability of '{R['binary_label']}' at {R['binary_x']} = {x_value:g}:")
for lv, pr in zip(levels, probs):
    print(f"   {g} = {lv}: {pr:.1%}")

## D. Odds ratios are not risk ratios
### 🧭 Concept
"OR = 2" is often reported as "twice as likely". That is only roughly true when the outcome is **rare**. Compare the raw risk ratio (RR) with the odds ratio between two groups.

In [ ]:
# 🔧 CHANGE THIS: which group to compare with the reference
compare_level = sorted(df[R["binary_group"]].dropna().unique())[-1]

yb, g = R["binary"], R["binary_group"]
dd = df.dropna(subset=[yb, g])
ref_level = sorted(dd[g].unique())[0]
p1 = dd.loc[dd[g] == compare_level, yb].mean()
p0 = dd.loc[dd[g] == ref_level, yb].mean()
print(f"Risk in {compare_level}: {p1:.1%}    Risk in {ref_level}: {p0:.1%}")
print(f"Risk ratio (RR): {p1 / p0:.2f}")
print(f"Odds ratio (OR): {(p1 / (1 - p1)) / (p0 / (1 - p0)):.2f}")

### ✅ Concept check
When is "OR ≈ RR" a reasonable shortcut?

<details><summary>Show answer</summary>

When the outcome is rare (roughly under 10%) in both groups. For common outcomes the OR is further from 1 than the RR, so "times as likely" overstates the effect.
</details>

## 🤖 Go deeper with AI
Three ways to keep learning about logistic regression: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain probability, odds and log-odds using 20 patients as an example. Then quiz me: ask me to convert between them three times, and correct me.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *For odds ratios of 1.5, 2.5 and 4, plot the corresponding risk ratio against the risk in the reference group, from 1% to 80%.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *Here is my logistic regression output: [paste]. Turn the odds ratios into predicted probabilities for two typical patients, and write one sentence a clinician could use.*

✔ **Check one claim yourself:** The risk ratio should be close to the odds ratio only when the reference risk is small (section D).

## 💬 Discuss with your partner
- How would you report the result of section B in a paper: odds ratio, risk difference, predicted probabilities?
- Which binary outcomes in your own work are common enough that OR ≠ RR matters?
- The `lung` binary outcome "lost weight" depends on how long patients were followed. When is a yes/no outcome really a hidden time-to-event outcome (Day 3)?